# 03 — Operations & Communication
**Agentic X | Studio 67** — Real components only. Run from the repository root.

1 — Project Setup & GitHub Connection
This cell connects Google Colab to the Agentic X GitHub repository, installs the required Python libraries, and verifies that the project files are available.

In [21]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/ProjectsSpaceHub/AgenticX.git"
ROOT = Path("/content/AgenticX")

if not ROOT.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(ROOT)],
        check=True
    )
else:
    if not (ROOT / ".git").exists():
        raise RuntimeError(
            "Project folder exists but is not a Git checkout"
        )

    status = subprocess.run(
        ["git", "-C", str(ROOT), "status", "--porcelain"],
        check=True,
        capture_output=True,
        text=True
    ).stdout.strip()

    if status:
        print("Local edits found. Skipping git pull.")
    else:
        subprocess.run(
            ["git", "-C", str(ROOT), "pull", "--ff-only"],
            check=True
        )

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "-q", "-r", str(ROOT / "requirements.txt")
    ],
    check=True
)

required_files = [
    "core/models.py",
    "core/environment.py",
    "agents/creative.py",
    "agents/finance.py",
    "agents/safety.py",
    "agents/operations.py",
    "core/communication.py",
    "core/negotiation.py",
    "data/AgenticX_150_Validated_Final.xlsx"
]

for file in required_files:
    assert (ROOT / file).is_file(), f"Missing: {file}"

print("Notebook 03 environment ready!")
print("Project path:", ROOT)

Local edits found. Skipping git pull.
Notebook 03 environment ready!
Project path: /content/AgenticX


2 — Import Libraries & Load Real Scenario
This cell imports the shared project modules and loads the synthetic dataset. It selects a Screen Failure case to test Operations and communication

In [22]:
import pandas as pd
from IPython.display import display

from core.environment import load_dataset, get_case
from core.models import AgentEvaluation, ACLMessage
from agents.finance import FinanceAgent
from agents.safety import SafetyAgent

cases_df, options_df = load_dataset()

assert len(cases_df) == 150
assert len(options_df) == 450

case, options = get_case("SCR-001")

assert len(options) == 3

print("Total cases:", len(cases_df))
print("Total options:", len(options_df))
print("Case ID:", case["case_id"])
print("Incident:", case["incident"])

display(
    pd.DataFrame(options)[
        ["option_id", "option_name",
         "duration_min", "cost_sar"]
    ]
)

print("Dataset validation passed!")

Total cases: 150
Total options: 450
Case ID: SCR-001
Incident: Main LED screen failed during final rehearsal


,option_id,option_name,duration_min,cost_sar
0,SCR-001-O1,Use backup LED,40,1290
1,SCR-001-O2,Rent replacement screen,70,5920
2,SCR-001-O3,Continue with projection,25,1210


Dataset validation passed!


3 — Configure Gemini API Key
This cell securely loads the Gemini API key from Google Colab Secrets and configures the language model used by Operations.

In [23]:
from google.colab import userdata

key = userdata.get("GOOGLE_API_KEY")

if not isinstance(key, str) or not key.strip():
    raise RuntimeError(
        "Add GOOGLE_API_KEY to Colab Secrets "
        "and enable notebook access."
    )

os.environ["GOOGLE_API_KEY"] = key.strip()
del key

os.environ["AGENTICX_MODEL"] = "gemini-3.8-flash"

print("Gemini API configured successfully!")
print("Model:", os.environ["AGENTICX_MODEL"])

Gemini API configured successfully!
Model: gemini-3.8-flash


4 — Build Operations LLM Agent
This cell implements the Operations Agent using Gemini and Python resource validation. The agent checks time, crew, backup equipment, venue access, and supplier availability before requesting an LLM evaluation.

In [24]:
import importlib

operations_code = r'''
"""Operations Agent — Gemini LLM and validated resource checks."""

import os
import math
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from core.models import AgentEvaluation


class OperationsOpinion(BaseModel):
    approved: bool
    score: float = Field(ge=0, le=100)
    reason: str


def valid_number(value):
    """Return a non-negative finite number, or None."""
    if value is None or isinstance(value, bool):
        return None

    try:
        number = float(value)

        if math.isfinite(number) and number >= 0:
            return number

    except (TypeError, ValueError, OverflowError):
        pass

    return None


def is_true(value):
    """Interpret supported boolean values safely."""
    if value is True or value == 1:
        return True

    if isinstance(value, str):
        return value.strip().lower() in (
            "true", "yes", "1"
        )

    return False


class OperationsAgent:
    name = "Operations"

    def __init__(self):
        if not os.getenv("GOOGLE_API_KEY"):
            raise RuntimeError(
                "GOOGLE_API_KEY is missing"
            )

        self.llm = ChatGoogleGenerativeAI(
            model=os.getenv(
                "AGENTICX_MODEL",
                "gemini-3.8-flash"
            ),
            temperature=0
        ).with_structured_output(
            OperationsOpinion,
            method="json_schema"
        )

    def evaluate(self, case, option):
        duration = valid_number(
            option.get("duration_min")
        )

        time_available = valid_number(
            case.get("time_left_min")
        )

        crew_required = valid_number(
            option.get("crew_required")
        )

        crew_available = valid_number(
            case.get("crew_count_available")
        )

        needs_crew = (
            is_true(option.get("needs_crew"))
            or (
                crew_required is not None
                and crew_required > 0
            )
        )

        checks = {
            "time": (
                duration is not None
                and time_available is not None
                and duration <= time_available
            ),

            "crew": (
                not needs_crew
                or (
                    crew_required is not None
                    and crew_required.is_integer()
                    and crew_required > 0
                    and crew_available is not None
                    and crew_available.is_integer()
                    and crew_required <= crew_available
                    and is_true(
                        case.get("crew_available")
                    )
                )
            ),

            "backup": (
                not is_true(
                    option.get("needs_backup")
                )
                or is_true(
                    case.get("backup_available")
                )
            ),

            "venue_access": (
                not is_true(
                    option.get("requires_venue_access")
                )
                or is_true(
                    case.get("venue_access")
                )
            ),

            "supplier": is_true(
                option.get("supplier_available")
            )
        }

        failed = [
            name
            for name, passed in checks.items()
            if not passed
        ]

        if failed:
            return AgentEvaluation(
                agent=self.name,
                option_id=str(option["option_id"]),
                approved=False,
                score=0,
                reason=(
                    "Operational constraints failed: "
                    + ", ".join(failed)
                ),
                blocking=True
            )

        prompt = (
            "You are the Operations Agent for Studio 67. "
            "Evaluate practical operational feasibility. "
            "The resource checks have passed. "
            "Use only the provided facts. "
            "Do not invent suppliers, staff, or equipment. "
            "Return approved, score (0-100), and reason. "
            f"CASE: {case!r} OPTION: {option!r}"
        )

        answer = self.llm.invoke(prompt)

        return AgentEvaluation(
            agent=self.name,
            option_id=str(option["option_id"]),
            approved=answer.approved,
            score=answer.score,
            reason=answer.reason,
            blocking=True
        )
'''

operations_path = ROOT / "agents" / "operations.py"

compile(
    operations_code,
    str(operations_path),
    "exec"
)

operations_path.write_text(
    operations_code,
    encoding="utf-8"
)

import agents.operations as operations_module
importlib.reload(operations_module)

operations_agent = operations_module.OperationsAgent()

assert operations_agent.name == "Operations"

print("Operations Agent initialized successfully!")
print("Python file:", operations_path)

Operations Agent initialized successfully!
Python file: /content/AgenticX/agents/operations.py


5 — Build ACL Communication System
This cell implements structured communication between agents using six ACL message types. It validates messages, links replies to requests, and maintains a communication history.

In [25]:
communication_code = r'''
"""Validated ACL communication system for Agentic X."""

from core.models import ACLMessage

VALID_PERFORMATIVES = {
    "REQUEST", "INFORM", "CFP",
    "PROPOSE", "ACCEPT", "REFUSE"
}


class MessageBus:
    def __init__(self, case_id):
        if (
            not isinstance(case_id, str)
            or not case_id.strip()
        ):
            raise ValueError(
                "case_id must be a non-empty string"
            )

        self.case_id = case_id
        self.messages = []

    def send(
        self, round, sender, receiver,
        performative, content, in_reply_to=None
    ):
        if type(round) is not int or round < 1:
            raise ValueError(
                "Round must be a positive integer"
            )

        for name, value in [
            ("sender", sender),
            ("receiver", receiver),
            ("content", content)
        ]:
            if (
                not isinstance(value, str)
                or not value.strip()
            ):
                raise ValueError(
                    f"{name} must be a non-empty string"
                )

        if performative not in VALID_PERFORMATIVES:
            raise ValueError(
                "Invalid ACL performative"
            )

        if in_reply_to is not None:
            referenced = next(
                (
                    m
                    for m in self.messages
                    if m.message_id == in_reply_to
                ),
                None
            )

            if referenced is None:
                raise ValueError(
                    "Referenced message does not exist"
                )

            if referenced.round > round:
                raise ValueError(
                    "Cannot reply to a future round"
                )

        message = ACLMessage(
            message_id=(
                f"{self.case_id}-M"
                f"{len(self.messages)+1:04d}"
            ),
            case_id=self.case_id,
            round=round,
            sender=sender,
            receiver=receiver,
            performative=performative,
            content=content,
            in_reply_to=in_reply_to
        )

        self.messages.append(message)
        return message

    def history(
        self, agent=None, performative=None
    ):
        messages = self.messages

        if agent is not None:
            messages = [
                m for m in messages
                if m.sender == agent
                or m.receiver == agent
                or m.receiver == "ALL"
            ]

        if performative is not None:
            messages = [
                m for m in messages
                if m.performative == performative
            ]

        return messages

    def to_records(self):
        return [
            msg.model_dump()
            for msg in self.messages
        ]
'''

communication_path = (
    ROOT / "core" / "communication.py"
)

compile(
    communication_code,
    str(communication_path),
    "exec"
)

communication_path.write_text(
    communication_code,
    encoding="utf-8"
)

import core.communication as communication_module
importlib.reload(communication_module)

MessageBus = communication_module.MessageBus

print("ACL Communication System initialized!")
print("Python file:", communication_path)

ACL Communication System initialized!
Python file: /content/AgenticX/core/communication.py


6 — Operations Agent PEAS & Type
This cell documents the Operations Agent using the PEAS framework and classifies it as a Hybrid Agent because it combines deterministic resource checks with LLM-based evaluation.

In [26]:
operations_peas = {
    "Performance Measure": (
        "Operational feasibility, deadline compliance, "
        "resource availability, and justified decisions"
    ),

    "Environment": (
        "Studio 67 event crisis management simulation"
    ),

    "Actuators": (
        "Evaluate options, approve or reject solutions, "
        "provide explanations, and send ACL messages"
    ),

    "Sensors": (
        "Incident details, remaining time, crew availability, "
        "backup equipment, suppliers, and venue access"
    )
}

peas_df = pd.DataFrame(
    operations_peas.items(),
    columns=["PEAS Component", "Description"]
)

display(peas_df)

assert len(peas_df) == 4

print("Agent: Operations")
print("Agent Type: Hybrid")
print(
    "Justification: Python resource validation "
    "combined with Gemini LLM evaluation."
)

,PEAS Component,Description
0,Performance Measure,"Operational feasibility, deadline compliance, ..."
1,Environment,Studio 67 event crisis management simulation
2,Actuators,"Evaluate options, approve or reject solutions,..."
3,Sensors,"Incident details, remaining time, crew availab..."


Agent: Operations
Agent Type: Hybrid
Justification: Python resource validation combined with Gemini LLM evaluation.


7 — Test Operational Constraints
This cell tests whether Operations correctly rejects solutions that violate resource constraints. Invalid or impossible options are rejected without calling the LLM.

In [27]:
test_option = options[0]

invalid_cases = [
    (
        "Missing Time",
        {**case, "time_left_min": None},
        test_option,
        "time"
    ),
    (
        "Negative Time",
        {**case, "time_left_min": -5},
        test_option,
        "time"
    ),
    (
        "Invalid Time",
        {**case, "time_left_min": "unknown"},
        test_option,
        "time"
    ),
    (
        "Missing Duration",
        case,
        {**test_option, "duration_min": None},
        "time"
    ),
    (
        "Missing Supplier",
        case,
        {**test_option, "supplier_available": None},
        "supplier"
    ),
    (
        "Insufficient Time",
        {**case, "time_left_min": 5},
        test_option,
        "time"
    ),
    (
        "Insufficient Crew",
        {**case, "crew_count_available": 0},
        {
            **test_option,
            "crew_required": 3,
            "needs_crew": True
        },
        "crew"
    ),
    (
        "Supplier Unavailable",
        case,
        {
            **test_option,
            "supplier_available": False
        },
        "supplier"
    )
]

constraint_results = []

for name, test_case, option, expected_failure in invalid_cases:

    result = operations_agent.evaluate(
        test_case, option
    )

    assert result.approved is False
    assert result.score == 0
    assert result.blocking is True
    assert expected_failure in result.reason

    constraint_results.append({
        "Test": name,
        "Approved": result.approved,
        "Reason": result.reason
    })

display(pd.DataFrame(constraint_results))

print(
    "All", len(constraint_results),
    "operational constraint tests passed!"
)

,Test,Approved,Reason
0,Missing Time,False,Operational constraints failed: time
1,Negative Time,False,Operational constraints failed: time
2,Invalid Time,False,Operational constraints failed: time
3,Missing Duration,False,Operational constraints failed: time
4,Missing Supplier,False,Operational constraints failed: supplier
5,Insufficient Time,False,Operational constraints failed: time
6,Insufficient Crew,False,Operational constraints failed: crew
7,Supplier Unavailable,False,Operational constraints failed: supplier


All 8 operational constraint tests passed!


8 — Test Operations Across Three Scenarios
This cell tests the three required scenarios: Screen Failure, Budget Overrun, and Safety Conflict. It checks whether Operations rejects solutions when the available execution time is zero.

In [28]:
scenario_results = []

for case_id in ["SCR-001", "BDG-001", "SFT-001"]:

    scenario_case, scenario_options = get_case(case_id)

    selected_option = next(
        option for option in scenario_options
        if float(option.get("duration_min") or 0) > 0
    )

    # Test override: zero minutes remaining
    test_case = scenario_case.copy()
    test_case["time_left_min"] = 0

    result = operations_agent.evaluate(
        test_case,
        selected_option
    )

    assert result.approved is False
    assert "time" in result.reason

    scenario_results.append({
        "Case ID": case_id,
        "Scenario": scenario_case["scenario"],
        "Option": selected_option["option_id"],
        "Original Time": scenario_case["time_left_min"],
        "Test Time": 0,
        "Approved": result.approved,
        "Reason": result.reason
    })

scenario_df = pd.DataFrame(scenario_results)

display(scenario_df)

assert len(scenario_df) == 3
assert not scenario_df["Approved"].any()

print("Three scenario deadline tests passed!")

,Case ID,Scenario,Option,Original Time,Test Time,Approved,Reason
0,SCR-001,Screen Failure,SCR-001-O1,100,0,False,Operational constraints failed: time
1,BDG-001,Budget Overrun,BDG-001-O1,80,0,False,Operational constraints failed: time
2,SFT-001,Safety Conflict,SFT-001-O1,60,0,False,Operational constraints failed: time


Three scenario deadline tests passed!


9 — Validate ACL Messages & Error Handling
This cell verifies that ACL messages are correctly linked, broadcast messages are visible to agents, and invalid messages are rejected.

In [29]:
acl_bus = MessageBus(case["case_id"])

# Broadcast a call for proposals
cfp = acl_bus.send(
    1, "Coordinator", "ALL",
    "CFP", "Evaluate available options"
)

# Request an evaluation
request = acl_bus.send(
    1, "Coordinator", "Operations",
    "REQUEST", "Assess operational feasibility"
)

# Reply to the request
reply = acl_bus.send(
    1, "Operations", "Coordinator",
    "INFORM", "Resources are being checked",
    in_reply_to=request.message_id
)

# Verify message history
history = acl_bus.history(agent="Operations")

assert all(
    message in history
    for message in [cfp, request, reply]
)

assert reply.in_reply_to == request.message_id
assert len(acl_bus.to_records()) == 3

# Test invalid messages
invalid_tests = [
    {
        "round": 0,
        "sender": "Coordinator",
        "receiver": "Operations",
        "performative": "REQUEST",
        "content": "Test"
    },
    {
        "round": 1,
        "sender": None,
        "receiver": "Operations",
        "performative": "REQUEST",
        "content": "Test"
    },
    {
        "round": 1,
        "sender": "Coordinator",
        "receiver": None,
        "performative": "REQUEST",
        "content": "Test"
    },
    {
        "round": 1,
        "sender": "Coordinator",
        "receiver": "Operations",
        "performative": "REQUEST",
        "content": None
    },
    {
        "round": 1,
        "sender": "Coordinator",
        "receiver": "Operations",
        "performative": "UNKNOWN",
        "content": "Test"
    },
    {
        "round": 1,
        "sender": "Coordinator",
        "receiver": "Operations",
        "performative": "INFORM",
        "content": "Test",
        "in_reply_to": "INVALID-ID"
    }
]

for test in invalid_tests:
    try:
        acl_bus.send(**test)
    except ValueError:
        continue

    raise AssertionError(
        f"Invalid message accepted: {test}"
    )

assert len(acl_bus.messages) == 3

display(pd.DataFrame(acl_bus.to_records())[
    ["message_id", "sender", "receiver",
     "performative", "in_reply_to"]
])

print("ACL communication validation passed!")
print("6 invalid-message tests passed!")

,message_id,sender,receiver,performative,in_reply_to
0,SCR-001-M0001,Coordinator,ALL,CFP,None
1,SCR-001-M0002,Coordinator,Operations,REQUEST,None
2,SCR-001-M0003,Operations,Coordinator,INFORM,SCR-001-M0002


ACL communication validation passed!
6 invalid-message tests passed!


10 — Test Real Gemini Operations Agent
This cell evaluates a real solution using Gemini. The agent returns a structured decision, score, and explanation based on the scenario and operational constraints.

In [30]:
print("Case:", case["case_id"])
print("Option:", test_option["option_name"])
print("Evaluating with Gemini...")

evaluation = operations_agent.evaluate(
    case,
    test_option
)

result = evaluation.model_dump()

display(pd.DataFrame([result]))

assert isinstance(evaluation, AgentEvaluation)

assert result["agent"] == "Operations"
assert result["option_id"] == test_option["option_id"]
assert isinstance(result["approved"], bool)
assert 0 <= result["score"] <= 100
assert result["reason"].strip()

print("Real Operations evaluation passed!")

Case: SCR-001
Option: Use backup LED
Evaluating with Gemini...


,agent,option_id,approved,score,reason,blocking
0,Operations,SCR-001-O1,True,95.0,Option is fully feasible: 40-minute duration f...,True


Real Operations evaluation passed!


11 — Agent Communication Using Real Evaluations
This cell demonstrates communication between Operations, Finance, and Safety. ACL messages are generated from actual agent evaluations rather than predefined dialogue.

In [31]:
# Evaluate the same option with Finance and Safety
finance_result = FinanceAgent().evaluate(
    case, test_option
)

safety_result = SafetyAgent().evaluate(
    case, test_option
)

agent_results = [
    evaluation,
    finance_result,
    safety_result
]

assert {
    result.agent for result in agent_results
} == {
    "Operations", "Finance", "Safety"
}

# Create the communication log
bus = MessageBus(case["case_id"])

bus.send(
    1, "Coordinator", "ALL", "CFP",
    f"Evaluate option {test_option['option_id']}"
)

# Request assessments and record responses
for agent_result in agent_results:

    request = bus.send(
        1,
        "Coordinator",
        agent_result.agent,
        "REQUEST",
        f"Evaluate {agent_result.option_id}"
    )

    bus.send(
        1,
        agent_result.agent,
        "Coordinator",
        "INFORM",
        agent_result.reason,
        in_reply_to=request.message_id
    )

    bus.send(
        1,
        agent_result.agent,
        "ALL",
        "ACCEPT" if agent_result.approved else "REFUSE",
        f"{agent_result.option_id}: {agent_result.reason}"
    )

# Operations proposes the option if approved
if evaluation.approved:
    bus.send(
        1,
        "Operations",
        "ALL",
        "PROPOSE",
        f"Propose {evaluation.option_id}: {evaluation.reason}"
    )

messages_df = pd.DataFrame(bus.to_records())

display(messages_df[
    ["round", "sender", "receiver",
     "performative", "content"]
])

assert all(
    message.case_id == case["case_id"]
    for message in bus.messages
)

assert len({
    message.message_id
    for message in bus.messages
}) == len(bus.messages)

print("Agent communication validation passed!")
print("Total ACL messages:", len(bus.messages))
print("Agents:", [result.agent for result in agent_results])
print("Final negotiation belongs to Notebook 04.")

,round,sender,receiver,performative,content
0,1,Coordinator,ALL,CFP,Evaluate option SCR-001-O1
1,1,Coordinator,Operations,REQUEST,Evaluate SCR-001-O1
2,1,Operations,Coordinator,INFORM,Option is fully feasible: 40-minute duration f...
3,1,Operations,ALL,ACCEPT,SCR-001-O1: Option is fully feasible: 40-minut...
4,1,Coordinator,Finance,REQUEST,Evaluate SCR-001-O1
5,1,Finance,Coordinator,INFORM,Committed plus option cost: 1290.00 SAR versus...
6,1,Finance,ALL,ACCEPT,SCR-001-O1: Committed plus option cost: 1290.0...
7,1,Coordinator,Safety,REQUEST,Evaluate SCR-001-O1
8,1,Safety,Coordinator,INFORM,All safety checks passed
9,1,Safety,ALL,ACCEPT,SCR-001-O1: All safety checks passed


Agent communication validation passed!
Total ACL messages: 11
Agents: ['Operations', 'Finance', 'Safety']
Final negotiation belongs to Notebook 04.


12 — Validate All Six ACL Message Types
This cell verifies that the communication system supports REQUEST, INFORM, CFP, PROPOSE, ACCEPT, and REFUSE. A separate operational constraint test is used to demonstrate a valid refusal.

In [35]:
protocol_bus = MessageBus(case["case_id"])

# CFP
protocol_bus.send(
    1, "Coordinator", "ALL",
    "CFP", f"Request proposals for {test_option['option_id']}"
)

# REQUEST
request = protocol_bus.send(
    1, "Coordinator", "Operations",
    "REQUEST", "Evaluate the proposed solution"
)

# INFORM
protocol_bus.send(
    1, "Operations", "Coordinator",
    "INFORM", evaluation.reason,
    in_reply_to=request.message_id
)

# PROPOSE
protocol_bus.send(
    1, "Operations", "ALL",
    "PROPOSE",
    f"Candidate option: {evaluation.option_id}"
)

# ACCEPT or REFUSE from real agent evaluations
for agent_result in [finance_result, safety_result]:

    protocol_bus.send(
        1,
        agent_result.agent,
        "Coordinator",
        "ACCEPT" if agent_result.approved else "REFUSE",
        f"{agent_result.option_id}: {agent_result.reason}"
    )

# Separate refusal test: zero remaining time
zero_time_case = case.copy()
zero_time_case["time_left_min"] = 0

zero_time_result = operations_agent.evaluate(
    zero_time_case,
    test_option
)

assert zero_time_result.approved is False

protocol_bus.send(
    1, "Operations", "Coordinator",
    "REFUSE",
    "Separate zero-time test: " + zero_time_result.reason
)

required_types = {
    "REQUEST", "INFORM", "CFP",
    "PROPOSE", "ACCEPT", "REFUSE"
}

observed_types = {
    message.performative
    for message in protocol_bus.messages
}

assert required_types.issubset(observed_types)

display(pd.DataFrame(protocol_bus.to_records())[
    ["sender", "receiver", "performative", "content"]
])

print("All six ACL message types validated!")
print("Types:", sorted(observed_types))
print("This is a protocol test, not final negotiation.")

,sender,receiver,performative,content
0,Coordinator,ALL,CFP,Request proposals for SCR-001-O1
1,Coordinator,Operations,REQUEST,Evaluate the proposed solution
2,Operations,Coordinator,INFORM,Option is fully feasible: 40-minute duration f...
3,Operations,ALL,PROPOSE,Candidate option: SCR-001-O1
4,Finance,Coordinator,ACCEPT,SCR-001-O1: Committed plus option cost: 1290.0...
5,Safety,Coordinator,ACCEPT,SCR-001-O1: All safety checks passed
6,Operations,Coordinator,REFUSE,Separate zero-time test: Operational constrain...


All six ACL message types validated!
Types: ['ACCEPT', 'CFP', 'INFORM', 'PROPOSE', 'REFUSE', 'REQUEST']
This is a protocol test, not final negotiation.


13 — Verify Integration Compatibility
This cell checks that the Operations Agent and communication system follow the shared project interfaces required by the previous and following notebooks.

In [32]:
import inspect

import agents.operations as ops_module
import core.communication as comm_module

from core.negotiation import NegotiationEngine
from core.runner import build_engine
from agents.operations import OperationsAgent

# Verify Python file paths
assert Path(ops_module.__file__).resolve() == (
    operations_path.resolve()
)

assert Path(comm_module.__file__).resolve() == (
    communication_path.resolve()
)

# Verify Gemini implementation
operations_source = operations_path.read_text(
    encoding="utf-8"
)

assert "ChatGoogleGenerativeAI" in operations_source
assert "ChatOpenAI" not in operations_source

# Verify communication interface
communication_source = communication_path.read_text(
    encoding="utf-8"
)

assert "def history" in communication_source
assert "def to_records" in communication_source

# Verify shared agent interface
agents = [
    OperationsAgent(),
    FinanceAgent(),
    SafetyAgent()
]

assert all(
    callable(getattr(agent, "evaluate", None))
    for agent in agents
)

# Verify Notebook 04 integration
engine_parameters = inspect.signature(
    NegotiationEngine.__init__
).parameters

assert all(
    name in engine_parameters
    for name in [
        "creative", "finance",
        "safety", "operations"
    ]
)

assert callable(NegotiationEngine.run)
assert callable(build_engine)

# Verify data models
assert isinstance(evaluation, AgentEvaluation)
assert isinstance(finance_result, AgentEvaluation)
assert isinstance(safety_result, AgentEvaluation)
assert isinstance(bus.messages[0], ACLMessage)

print("Operations Agent interface: PASS")
print("ACL Communication interface: PASS")
print("Notebook 01 and 02 compatibility: PASS")
print("Notebook 04 interface compatibility: PASS")
print("Shared data models: PASS")
print("All integration interface checks passed!")

Operations Agent interface: PASS
ACL Communication interface: PASS
Notebook 01 and 02 compatibility: PASS
Notebook 04 interface compatibility: PASS
Shared data models: PASS
All integration interface checks passed!


14 — Export Python Files for GitHub
This cell packages the final Operations and ACL communication Python files for manual upload to GitHub. The notebook itself must be downloaded separately as an .ipynb file.

In [34]:
from zipfile import ZipFile
from google.colab import files

archive_path = Path(
    "/content/AgenticX_Notebook03_Final_Python.zip"
)

with ZipFile(archive_path, "w") as archive:

    for file_path in [
        operations_path,
        communication_path
    ]:
        assert file_path.is_file()

        archive.write(
            file_path,
            arcname=file_path.relative_to(ROOT)
        )

        print(
            "Included:",
            file_path.relative_to(ROOT)
        )

print("Final Python files packaged successfully!")
print("ZIP file:", archive_path)

files.download(str(archive_path))

Included: agents/operations.py
Included: core/communication.py
Final Python files packaged successfully!
ZIP file: /content/AgenticX_Notebook03_Final_Python.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Conclusion

Notebook 03 implements a Hybrid Operations Agent using Gemini LLM reasoning and deterministic Python resource validation. The agent evaluates operational feasibility based on time, staff availability, backup equipment, venue access, and supplier availability.

The notebook also implements structured ACL communication with six message types, request–reply linkage, broadcast messaging, history tracking, and input validation. Operational constraint tests and communication tests demonstrate the system's behavior under valid and invalid conditions.

The final decision-making and negotiation process will be implemented in Notebook 04, while full scenario execution, evaluation, and LangSmith integration are handled in Notebook 05.